In [1]:
from dbrepo.RestClient import RestClient
from dotenv import load_dotenv
import os 
from dbrepo.api.dto import CreateView
from dbrepo.api.dto import CreateView, Subset, SubsetColumn, Join
from dbrepo.api.dto import JoinType

load_dotenv()
password = os.getenv("DBREPO_PASS")
username = os.getenv("DBREPO_USER")
client = RestClient("https://test.dbrepo.tuwien.ac.at/", username=username, password=password)

containers = client.get_containers()
print(containers)

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [2]:
DB_ID = os.getenv("DB_ID") #"cf27a11d-58e5-4693-856c-e8f3527e3394"

In [3]:
df = client.get_database(DB_ID)
db_tables = client.get_tables(DB_ID)

In [4]:
tab_name_to_id = dict()
for i in db_tables:
    tab_name_to_id[i.name] = i.id

In [5]:
tab_name_to_id

{'wastewater_data': 'c91317e1-3568-451e-a496-ff76453b353f',
 'gdp_data': 'fb47cfcc-9802-4b27-9665-e9b3b2faa756',
 'city_map': '56f561d5-5d48-41cf-983d-b9404233aea5'}

In [6]:
tab_id_and_col_to_col_id = dict()

for tab_id in tab_name_to_id.values():
    table = client.get_table(DB_ID, tab_id)
    for col in table.columns:
        tab_id_and_col_to_col_id[(tab_id, col.name)] = col.id

In [7]:
table = client.get_table(DB_ID, tab_name_to_id["city_map"])
for col in table.columns:
    print(col)

id='989ce469-d5d1-49eb-a144-5e691a0626b7' name='nuts_code' database_id='5cde660e-153a-4bff-8e41-69e87cda399d' table_id='56f561d5-5d48-41cf-983d-b9404233aea5' ord=0 internal_name='nuts_code' is_null_allowed=False type=<ColumnType.VARCHAR: 'varchar'> alias=None description=None size=5 d=None mean=4.9588 median=4.9588 concept=None unit=None concept_uri='http://purl.org/linked-data/sdmx/2009/dimension#refArea' unit_uri='None' enums=[] sets=[] index_length=None length=None data_length=None max_data_length=None num_rows=None val_min=None val_max=None std_dev=0.2942
id='a3ec313a-5824-4480-86fa-3e759b8610e9' name='city_name' database_id='5cde660e-153a-4bff-8e41-69e87cda399d' table_id='56f561d5-5d48-41cf-983d-b9404233aea5' ord=1 internal_name='city_name' is_null_allowed=False type=<ColumnType.VARCHAR: 'varchar'> alias=None description=None size=100 d=None mean=8.4941 median=8.4941 concept=None unit=None concept_uri='http://purl.obolibrary.org/obo/NCIT_C95378' unit_uri='None' enums=[] sets=[] in

# Create View 1: Summary of drugs in water

In [8]:
wtable_id = tab_name_to_id["wastewater_data"]
df_city_summary_view = CreateView(
    name="ww_city_year_drug_summary",
    description="City-level aggregated wastewater indicators per year",
    is_public=True,
    is_schema_public=True,
    query=Subset(
        datasource_ids=[wtable_id],  # wastewater_data table ID

        columns=[
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "city_name")],
                alias="city_name"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "ref_year")],
                alias="ref_year"
            ),

            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "daily_mean_concentration")],
                aggregation="avg",
                alias="avg_daily_mean"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "daily_mean_concentration")],
                aggregation="max",
                alias="max_daily_mean"
            ),

            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "metabolite_name")],
                aggregation="count_distinct",
                alias="metabolite_count"
            )
        ],

        joins=None,
        filters=None,
        orders=None
    )
)


In [9]:
response = client._wrapper(
    method="post",
    url=f"/api/v1/database/{DB_ID}/view",
    payload=df_city_summary_view
)

print(response.status_code)
print(response.text)

201
{"id":"8550c148-db32-475c-8be6-d55e34782949","name":"ww_city_year_drug_summary","query":"select `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`daily_mean_concentration` as `avg_daily_mean`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`metabolite_name` as `metabolite_count`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` as `city_name`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`ref_year` as `ref_year` from `wastewater_data`","database_id":"5cde660e-153a-4bff-8e41-69e87cda399d","internal_name":"ww_city_year_drug_summary","is_public":true,"is_schema_public":true,"initial_view":false,"query_hash":"dbef9b83967f49b0b0bc0d988b47737a58c059d615829758cee955efc6b36a87","owned_by":"data_stewardship_group20"}


# Create View 2: Join all

In [10]:
table = client.get_table(DB_ID, tab_name_to_id["city_map"])
print(table)  # or the UUID of the table
for col in table.columns:
    print(col.id, col.name)

id='56f561d5-5d48-41cf-983d-b9404233aea5' database_id='5cde660e-153a-4bff-8e41-69e87cda399d' name='city_map' owner=UserBrief(username='data_stewardship_group20', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) columns=[Column(id='989ce469-d5d1-49eb-a144-5e691a0626b7', name='nuts_code', database_id='5cde660e-153a-4bff-8e41-69e87cda399d', table_id='56f561d5-5d48-41cf-983d-b9404233aea5', ord=0, internal_name='nuts_code', is_null_allowed=False, type=<ColumnType.VARCHAR: 'varchar'>, alias=None, description=None, size=5, d=None, mean=4.9588, median=4.9588, concept=None, unit=None, concept_uri='http://purl.org/linked-data/sdmx/2009/dimension#refArea', unit_uri='None', enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=0.2942), Column(id='a3ec313a-5824-4480-86fa-3e759b8610e9', name='city_name', database_id='5cde660e-153a-4bff-8e41-69e87cda399d', table_id='56f561d5-

In [11]:
table = client.get_table(DB_ID, tab_name_to_id["gdp_data"])
print(table) 
for col in table.columns:
    print(col.id, col.name)

id='fb47cfcc-9802-4b27-9665-e9b3b2faa756' database_id='5cde660e-153a-4bff-8e41-69e87cda399d' name='gdp_data' owner=UserBrief(username='data_stewardship_group20', id=None, name=None, orcid=None, qualified_name=None, given_name=None, family_name=None) columns=[Column(id='981e7393-e469-4060-8d52-604ff8d96b87', name='nuts_code', database_id='5cde660e-153a-4bff-8e41-69e87cda399d', table_id='fb47cfcc-9802-4b27-9665-e9b3b2faa756', ord=0, internal_name='nuts_code', is_null_allowed=False, type=<ColumnType.VARCHAR: 'varchar'>, alias=None, description='5-character NUTS-3 administrative code (e.g. AT221): https://ec.europa.eu/eurostat/web/nuts', size=5, d=None, mean=5.0, median=5.0, concept=None, unit=None, concept_uri=None, unit_uri=None, enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=0.0), Column(id='c06a9d89-1c79-4946-949d-85de8da323a6', name='ref_year', database_id='5cde660e-153a-4bff-8e41-69e87cda39

In [12]:
df_ml_view = CreateView(
    name="drug_gdp_features_view",
    description="ML-ready dataset joining wastewater measurements with GDP via city-NUTS mapping",
    is_public=True,
    is_schema_public=True,
    query=Subset(
        datasource_ids=[
            wtable_id  # wastewater_data
        ],

        columns=[
            # wastewater_data
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "city_name")],  # city_name
                alias="city_name"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "ref_year")],  # ref_year
                alias="ref_year"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "metabolite_name")],  # metabolite_name
                alias="metabolite_name"
            ),
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(wtable_id, "daily_mean_concentration")],  # daily_mean_concentration
                alias="daily_mean"
            ),

            # city_map.nuts_code
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(tab_name_to_id["city_map"], "nuts_code")],  # nuts_code in city_map
                alias="nuts_code",
                #join_alias="m"
            ),

            # gdp_data.gdp
            SubsetColumn(
                id=tab_id_and_col_to_col_id[(tab_name_to_id["gdp_data"], "gdp")],  
                alias="gdp",
                #join_alias="g"
            )
        ],

        joins=[
            # wastewater_data → city_map
            Join(
                type=JoinType.INNER,
                datasource_id=tab_name_to_id["city_map"],
                #alias="m",
                conditionals=[
                    {
                        "column_id": tab_id_and_col_to_col_id[(wtable_id, "city_name")],        # wastewater.city_name
                        "foreign_column_id": tab_id_and_col_to_col_id[(tab_name_to_id["city_map"], "city_name")] # city_map.city_name
                    }
                ]
            ),

            # city_map → gdp_data
            Join(
                type=JoinType.INNER,
                datasource_id=tab_name_to_id["gdp_data"],
                #alias="g",
                conditionals=[
                    {
                        "foreign_column_id": tab_id_and_col_to_col_id[(tab_name_to_id["city_map"], "nuts_code")],        # city_map.nuts_code
                        "column_id": tab_id_and_col_to_col_id[(tab_name_to_id["gdp_data"], "nuts_code")] # gdp.nuts_code
                    }
                ]
            )
                    ],

        filters=None,
        orders=None
    )
)



In [13]:
response = client._wrapper(
    method="post",
    url=f"/api/v1/database/{DB_ID}/view",
    payload=df_ml_view
)

print(response.status_code)
print(response.text)

201
{"id":"6a6080f4-4117-4201-af05-876bf9eb05d5","name":"drug_gdp_features_view","query":"select `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`daily_mean_concentration` as `daily_mean`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`metabolite_name` as `metabolite_name`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`ref_year` as `ref_year`, `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code` as `nuts_code`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` as `city_name`, `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`gdp` as `gdp` from `wastewater_data` join `city_map` on `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`city_name` join `gdp_data` on `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`nuts_code` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code`","database_id":"5cde660e-153a-4bff-8e41-69e87cda399d","internal_